In [1]:
# Import the pandas library for data manipulation and analysis.
import pandas as pd
from Bio.Restriction import RestrictionBatch, Analysis, CommOnly
# Import the Seq object from Biopython for handling biological sequences.
from Bio.Seq import Seq

# Load sequences from Phase 3
df = pd.read_csv("results/fh_sequences.csv")
print(f"Loaded {len(df)} variants with sequences")

# CommOnly = all commercially available enzymes (~200 enzymes)
# This is what real labs can actually order and use
rb = RestrictionBatch(CommOnly)
print(f"Restriction enzymes to test: {len(rb)}")

Loaded 27 variants with sequences
Restriction enzymes to test: 623


In [2]:
def find_informative_enzymes(wt_seq, mut_seq, rb):
    """
    Compare restriction sites in WT vs mutant sequence.
    Returns list of enzymes that produce different cut patterns.
    """
    wt  = Seq(wt_seq)
    mut = Seq(mut_seq)
    
    wt_analysis  = Analysis(rb, wt,  linear=True)
    mut_analysis = Analysis(rb, mut, linear=True)
    
    wt_results  = wt_analysis.full()
    mut_results = mut_analysis.full()
    
    informative = []
    
    for enzyme in rb:
        wt_cuts  = list(wt_results[enzyme])
        mut_cuts = list(mut_results[enzyme])
        
        # Skip if pattern is identical
        if wt_cuts == mut_cuts:
            continue
        
        # Determine what changed
        if len(wt_cuts) > len(mut_cuts):
            change_type = "site_destroyed"
        elif len(wt_cuts) < len(mut_cuts):
            change_type = "site_created"
# Handle cases where sequences are missing.
        else:
            change_type = "position_shifted"
        
        # Calculate fragment sizes
        wt_frags  = cuts_to_fragments(wt_seq,  wt_cuts)
        mut_frags = cuts_to_fragments(mut_seq, mut_cuts)
        
        # Calculate how different the patterns look on a gel
        # Bigger difference = easier to read
        max_wt  = max(wt_frags)  if wt_frags  else 0
        max_mut = max(mut_frags) if mut_frags else 0
        frag_diff = abs(max_wt - max_mut)
        
        informative.append({
            "enzyme":       str(enzyme),
            "change_type":  change_type,
            "wt_cuts":      wt_cuts,
            "mut_cuts":     mut_cuts,
            "wt_frags":     wt_frags,
            "mut_frags":    mut_frags,
            "frag_diff_bp": frag_diff,
        })
    
    # Sort by fragment difference — most readable patterns first
    informative.sort(key=lambda x: x["frag_diff_bp"], reverse=True)
    return informative


def cuts_to_fragments(sequence, cut_positions):
    """
    Convert cut positions to fragment sizes in bp.
    """
    if not cut_positions:
        return [len(sequence)]
    
    boundaries = [0] + sorted(cut_positions) + [len(sequence)]
    return [boundaries[i+1] - boundaries[i] 
            for i in range(len(boundaries)-1)]


print("Functions defined — ready to scan")

Functions defined — ready to scan


In [3]:
all_results = []

for i, row in df.iterrows():
    print(f"Scanning {i+1}/27: {row['gene']} {row['cdna_change']}...", end=" ")
    
    informative = find_informative_enzymes(
        row["wt_seq"], 
        row["mut_seq"], 
        rb
    )
    
    if informative:
        # Take the best enzyme (highest fragment difference)
        best = informative[0]
        print(f"FOUND {len(informative)} enzymes — best: {best['enzyme']} "
              f"(diff: {best['frag_diff_bp']}bp)")
        
        all_results.append({
            "gene":            row["gene"],
            "cdna_change":     row["cdna_change"],
            "protein_change":  row["protein_change"],
            "chrom":           row["chrom"],
            "pos":             row["pos"],
            "rflp_detectable": True,
            "n_enzymes":       len(informative),
            "best_enzyme":     best["enzyme"],
            "change_type":     best["change_type"],
            "wt_frags":        str(best["wt_frags"]),
            "mut_frags":       str(best["mut_frags"]),
            "frag_diff_bp":    best["frag_diff_bp"],
            "all_enzymes":     str([e["enzyme"] for e in informative[:5]]),
        })
# Handle cases where sequences are missing.
    else:
        print("none found")
        all_results.append({
            "gene":            row["gene"],
            "cdna_change":     row["cdna_change"],
            "protein_change":  row["protein_change"],
            "chrom":           row["chrom"],
            "pos":             row["pos"],
            "rflp_detectable": False,
            "n_enzymes":       0,
            "best_enzyme":     "",
            "change_type":     "",
            "wt_frags":        "",
            "mut_frags":       "",
            "frag_diff_bp":    0,
            "all_enzymes":     "",
        })

df_results = pd.DataFrame(all_results)

print(f"\n{'='*50}")
print(f"SCAN COMPLETE")
print(f"{'='*50}")
print(f"Total variants scanned:      {len(df_results)}")
print(f"RFLP detectable:             {df_results['rflp_detectable'].sum()}")
print(f"Not detectable:              {(~df_results['rflp_detectable']).sum()}")
print(f"\nDetectable variants:")
print(df_results[df_results['rflp_detectable']][
    ["gene","cdna_change","best_enzyme","change_type","frag_diff_bp"]
])

Scanning 1/27: LDLR c.408C>G... FOUND 12 enzymes — best: EcoT38I (diff: 295bp)
Scanning 2/27: LDLR c.155G>T... FOUND 6 enzymes — best: PfoI (diff: 299bp)
Scanning 3/27: LDLR c.457T>C... FOUND 8 enzymes — best: GsuI (diff: 176bp)
Scanning 4/27: LDLR c.1840T>G... FOUND 3 enzymes — best: Hpy99I (diff: 299bp)
Scanning 5/27: LDLR c.1054T>G... FOUND 10 enzymes — best: Eam1104I (diff: 292bp)
Scanning 6/27: LDLR c.1961T>G... FOUND 6 enzymes — best: Eam1104I (diff: 294bp)
Scanning 7/27: LDLR c.1865A>T... FOUND 6 enzymes — best: EcoRV (diff: 298bp)
Scanning 8/27: LDLR c.1724T>A... FOUND 1 enzymes — best: MnlI (diff: 0bp)
Scanning 9/27: LDLR c.1074T>G... FOUND 4 enzymes — best: MspJI (diff: 0bp)
Scanning 10/27: LDLR c.986G>C... FOUND 13 enzymes — best: BsrDI (diff: 292bp)
Scanning 11/27: LDLR c.648T>G... FOUND 12 enzymes — best: GsaI (diff: 159bp)
Scanning 12/27: LDLR c.398A>G... FOUND 23 enzymes — best: BsaJI (diff: 64bp)
Scanning 13/27: LDLR c.483C>G... FOUND 28 enzymes — best: PaeI (diff: 299b

In [4]:
# Save full results
df_results.to_csv("results/fh_restriction_analysis.csv", index=False)

# Quality tiers based on fragment difference
def assign_quality(frag_diff):
    if frag_diff >= 150:
        return "excellent"   # Very clear band difference on gel
    elif frag_diff >= 50:
        return "good"        # Readable with care
    elif frag_diff > 0:
        return "poor"        # Hard to distinguish on gel
# Handle cases where sequences are missing.
    else:
        return "unusable"    # No size difference — not practical

df_results["gel_quality"] = df_results["frag_diff_bp"].apply(assign_quality)

# Exit inner loop once a relevant cut is found.
print("Quality breakdown:")
print(df_results["gel_quality"].value_counts())
print()

print("Excellent/Good candidates (practical for real lab use):")
df_good = df_results[df_results["gel_quality"].isin(["excellent","good"])]
print(df_good[["gene","cdna_change","best_enzyme",
               "change_type","frag_diff_bp","gel_quality"]])

print(f"\nTotal practical candidates: {len(df_good)}")

Quality breakdown:
gel_quality
excellent    21
unusable      2
good          2
poor          2
Name: count, dtype: int64

Excellent/Good candidates (practical for real lab use):
    gene                   cdna_change best_enzyme     change_type  \
0   LDLR                      c.408C>G     EcoT38I    site_created   
1   LDLR                      c.155G>T        PfoI    site_created   
2   LDLR                      c.457T>C        GsuI    site_created   
3   LDLR                     c.1840T>G      Hpy99I    site_created   
4   LDLR                     c.1054T>G    Eam1104I    site_created   
5   LDLR                     c.1961T>G    Eam1104I  site_destroyed   
6   LDLR                     c.1865A>T       EcoRV  site_destroyed   
9   LDLR                      c.986G>C       BsrDI  site_destroyed   
10  LDLR                      c.648T>G        GsaI    site_created   
11  LDLR                      c.398A>G       BsaJI    site_created   
12  LDLR                      c.483C>G        PaeI  

In [5]:
# Save the good candidates
df_good.to_csv("results/fh_rflp_candidates.csv", index=False)

# Save the full results with quality ratings
df_results.to_csv("results/fh_restriction_analysis.csv", index=False)

print("Files saved:")
print("  results/fh_restriction_analysis.csv — all 27 variants with quality ratings")
print("  results/fh_rflp_candidates.csv      — 23 practical candidates")
print()
print("Key findings for your paper:")
print(f"  Variants scanned:              27")
print(f"  RFLP detectable:               27 (100%)")
print(f"  Excellent gel quality:         21 (78%)")
print(f"  Good gel quality:               2 (7%)")
print(f"  Poor/unusable:                  4 (15%)")
print()
# Exit inner loop once a relevant cut is found.
print("Gene breakdown of excellent/good candidates:")
print(df_good["gene"].value_counts())
print()
print("Most common enzymes:")
print(df_good["best_enzyme"].value_counts().head(5))
print()
print("Site created vs destroyed:")
print(df_good["change_type"].value_counts())
print()
print("Phase 4 complete!")

Files saved:
  results/fh_restriction_analysis.csv — all 27 variants with quality ratings
  results/fh_rflp_candidates.csv      — 23 practical candidates

Key findings for your paper:
  Variants scanned:              27
  RFLP detectable:               27 (100%)
  Excellent gel quality:         21 (78%)
  Good gel quality:               2 (7%)
  Poor/unusable:                  4 (15%)

Gene breakdown of excellent/good candidates:
gene
LDLR    18
APOB     5
Name: count, dtype: int64

Most common enzymes:
best_enzyme
Eam1104I    3
EcoT38I     2
PfoI        1
GsuI        1
Hpy99I      1
Name: count, dtype: int64

Site created vs destroyed:
change_type
site_created      12
site_destroyed    11
Name: count, dtype: int64

Phase 4 complete!
